## Outline

- DF for minute features at date grain (Done)
- DF for daily features at date grain (Done)
- DF for returns over 1, 3 and 5 days (Done)
- Simple logistic, rfc and xgb models for daily alone, min alone and then combined
- Permutation importance
- Chart over rolling 5 days for 25 iterations, aka 6 months

In [1]:
import min_features, daily_return
import importlib
import pandas as pd

importlib.reload(min_features)
importlib.reload(daily_return)

df_min = min_features.min_features()
returns = [1, 3, 5]
df_daily = daily_return.pull_daily('QQQ', returns) 

df_main = pd.merge(df_min, df_daily, how='inner', on='Date')
df_main = df_main.sort_values(by='Date', ascending=False)

return_cols = df_main.columns[df_main.columns.str.contains("Return_")].to_list()
daily_cols = [
    c for c in df_daily.iloc[:, 1:].columns
    if "return" not in c.lower()
]
min_cols = df_min.iloc[:, 1:].columns.to_list()

# Multi Model Baseline Testing

In [5]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import balanced_accuracy_score, f1_score, accuracy_score
import warnings
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
warnings.filterwarnings("ignore", message="y_pred contains classes not in y_true")
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# -----------------------------
# Models
# -----------------------------
models = {
    "logistic": LogisticRegression(max_iter=1000, random_state=42),
    "linear_svm": LinearSVC(dual=False, random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "grad_boost": GradientBoostingClassifier(random_state=42),
    "naive_bayes": GaussianNB(),
}

# -----------------------------
# Helpers
# -----------------------------
def _compute_dist(y):
    """Distribution stats for y in {0,1}."""
    n = int(len(y))
    n_pos = int((y == 1).sum())
    n_neg = int((y == 0).sum())
    return {
        "test_n": n,
        "test_pos_n": n_pos,
        "test_neg_n": n_neg,
        "test_pos_frac": (n_pos / n) if n else np.nan,
        "test_neg_frac": (n_neg / n) if n else np.nan,
    }

def walkback_runs(
    df,
    feature_cols,
    target_col,
    *,
    date_col="Date",
    train_years=6,
    test_days=5,
    step_days=5,
    runs=20,
    horizon_days=1,        # r (used for purge)
    purge_days=None,       # defaults to horizon_days
    fill_inf=0.0,
):
    """
    Deployment-aligned evaluation:
      - For each run, take a 5-day OOT test window stepping back by 5 days.
      - Train on the prior N years (fixed-length window) ending right before test.
      - Purge 'purge_days' from the end of train to avoid overlap leakage for forward-return labels.
      - Score ONLY on the OOT test window (distribution + metrics).
    Returns: long DataFrame with one row per (feature_set/run/model).
    """
    dfw = df.sort_values("Date").reset_index(drop=True).copy()

    # Drop any accidental return cols from features (belt+suspenders)
    safe_feature_cols = [c for c in feature_cols if "Return" not in c]

    # Basic numeric cleaning
    dfw[safe_feature_cols] = dfw[safe_feature_cols].replace([np.inf, -np.inf], fill_inf)

    n = len(dfw)
    train_size = 245 * int(train_years)
    test_size = int(test_days)
    step = int(step_days)
    purge = int(purge_days) if purge_days is not None else int(horizon_days)

    X_all = dfw[safe_feature_cols].to_numpy()
    #y_all = _to_binary(dfw[target_col].to_numpy())
    y_all = dfw[target_col].to_numpy()
    dates = dfw[date_col].to_numpy() if date_col in dfw.columns else None

    rows = []

    for k in range(runs):
        test_end = n - k * step
        test_start = test_end - test_size
        if test_start < 0:
            break

        # Purge at boundary so train labels/features don't overlap test horizon
        train_end = test_start - purge
        train_start = train_end - train_size
        if train_start < 0 or train_end <= train_start:
            break

        print(
        f"Run {k+1}/{runs} | "
        f"Train: {dates[train_start]} → {dates[train_end-1]} | "
        f"Test: {dates[test_start]} → {dates[test_end-1]} | "
        f"Train_n={train_end-train_start} | Test_n={test_end-test_start}"
        )

        X_train = X_all[train_start:train_end]
        y_train = y_all[train_start:train_end]
        X_test = X_all[test_start:test_end]
        y_test = y_all[test_start:test_end]

        # OOT distribution (test only)
        dist = _compute_dist(y_test)

        for model_name, model in models.items():
            m = clone(model)
            m.fit(X_train, y_train)
            preds = m.predict(X_test)

            rows.append({
                "run": k + 1,
                "model": model_name,
                "bal_acc": float(balanced_accuracy_score(y_test, preds)),
                "f1": float(f1_score(y_test, preds, zero_division=0)),
                "acc": float(accuracy_score(y_test, preds)),
                **dist,
                "train_n": int(len(y_train)),
                "train_start": dates[train_start] if dates is not None else train_start,
                "train_end": dates[train_end - 1] if dates is not None else train_end - 1,
                "test_start": dates[test_start] if dates is not None else test_start,
                "test_end": dates[test_end - 1] if dates is not None else test_end - 1,
                "purge_days": purge,
                "train_years": train_years,
                "test_days": test_days,
                "step_days": step_days,
                "horizon_days": horizon_days,
                "n_features": len(safe_feature_cols),
            })

    return pd.DataFrame(rows)

# -----------------------------
# Run grid (feature sets x horizon x train_years, etc.)
# -----------------------------
column_sets = [daily_cols, min_cols, daily_cols + min_cols]
names = ["daily", "minute", "daily+minute"]

returns = [1, 3, 5]  # add 3,5,etc later
train_years_grid = [3, 5, 7]  # could be [3,4,5,6]
runs = 30
test_days = 5
step_days = 5

results_all = []

for feature_cols, feat_name in zip(column_sets, names):
    for r in returns:
        target_col = f"Return_{r}"

        for train_years in train_years_grid:
            df_scores = walkback_runs(
                df=df_main,
                feature_cols=feature_cols,
                target_col=target_col,
                date_col="Date",
                train_years=train_years,
                test_days=test_days,
                step_days=step_days,
                runs=runs,
                horizon_days=r,
                purge_days=r,   # purge = horizon (safe default)
                fill_inf=0.0,
            )

            df_scores["feature_set"] = feat_name
            df_scores["horizon"] = r

            results_all.append(df_scores)

results_df = pd.concat(results_all, ignore_index=True)

# -----------------------------
# Simple summaries you’ll actually use
# -----------------------------
# mean/std across the 20 OOT runs (deployment-aligned)
summary_df = (
    results_df
    .groupby(["feature_set", "horizon", "train_years", "model"], as_index=False)
    .agg(
        bal_acc_mean=("bal_acc", "mean"),
        bal_acc_std=("bal_acc", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        acc_mean=("acc", "mean"),
        acc_std=("acc", "std"),
        test_pos_frac_mean=("test_pos_frac", "mean"),
        test_pos_frac_min=("test_pos_frac", "min"),
        test_pos_frac_max=("test_pos_frac", "max"),
        n_runs=("run", "nunique"),
    )
    .sort_values(["feature_set", "horizon", "bal_acc_mean"], ascending=[True, True, False])
)

# best model per feature_set/horizon/train_years
best_df = summary_df.groupby(["feature_set", "horizon", "train_years"], as_index=False).head(1)


Run 1/30 | Train: 2022-12-28 → 2025-12-11 | Test: 2025-12-15 → 2025-12-19 | Train_n=735 | Test_n=5
Run 2/30 | Train: 2022-12-20 → 2025-12-04 | Test: 2025-12-08 → 2025-12-12 | Train_n=735 | Test_n=5
Run 3/30 | Train: 2022-12-13 → 2025-11-25 | Test: 2025-12-01 → 2025-12-05 | Train_n=735 | Test_n=5
Run 4/30 | Train: 2022-12-06 → 2025-11-18 | Test: 2025-11-20 → 2025-11-26 | Train_n=735 | Test_n=5
Run 5/30 | Train: 2022-11-29 → 2025-11-11 | Test: 2025-11-13 → 2025-11-19 | Train_n=735 | Test_n=5


KeyboardInterrupt: 

# Top Models Further Testing

In [ ]:
"""
rets = df_ph[ret_pct_col]
neg, pos = rets[rets < 0], rets[rets > 0]

neg_cut = neg.nlargest(max(1, int(len(neg) * 0.05))).min()
pos_cut = pos.nsmallest(max(1, int(len(pos) * 0.05))).max()
filtered = df_ph[(rets < neg_cut) | (rets > pos_cut)].copy()
"""

In [14]:
from xgboost import XGBClassifier

# -----------------------------
# Models
# -----------------------------
models = {
    "xgboost": XGBClassifier(n_estimators=400, random_state=42, n_jobs=-1),
    "random_forest": RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1),
}

# -----------------------------
# Helpers
# -----------------------------
def _compute_dist(y):
    """Distribution stats for y in {0,1}."""
    n = int(len(y))
    n_pos = int((y == 1).sum())
    n_neg = int((y == 0).sum())
    return {
        "test_n": n,
        "test_pos_n": n_pos,
        "test_neg_n": n_neg,
        "test_pos_frac": (n_pos / n) if n else np.nan,
        "test_neg_frac": (n_neg / n) if n else np.nan,
    }

def walkback_runs(
    df,
    feature_cols,
    target_col,
    *,
    date_col="Date",
    train_years=6,
    test_days=5,
    step_days=5,
    runs=20,
    horizon_days=1,        # r (used for purge)
    purge_days=None,       # defaults to horizon_days
    fill_inf=0.0,
):
    """
    Deployment-aligned evaluation:
      - For each run, take a 5-day OOT test window stepping back by 5 days.
      - Train on the prior N years (fixed-length window) ending right before test.
      - Purge 'purge_days' from the end of train to avoid overlap leakage for forward-return labels.
      - Score ONLY on the OOT test window (distribution + metrics).
    Returns: long DataFrame with one row per (feature_set/run/model).
    """
    dfw = df.sort_values("Date").reset_index(drop=True).copy()

    # Drop any accidental return cols from features (belt+suspenders)
    safe_feature_cols = [c for c in feature_cols if "Return" not in c]

    # Basic numeric cleaning
    dfw[safe_feature_cols] = dfw[safe_feature_cols].replace([np.inf, -np.inf], fill_inf)

    n = len(dfw)
    train_size = 245 * int(train_years)
    test_size = int(test_days)
    step = int(step_days)
    purge = int(purge_days) if purge_days is not None else int(horizon_days)

    X_all = dfw[safe_feature_cols].to_numpy()
    #y_all = _to_binary(dfw[target_col].to_numpy())
    y_all = dfw[target_col].to_numpy()
    dates = dfw[date_col].to_numpy() if date_col in dfw.columns else None

    rows = []

    for k in range(runs):
        test_end = n - k * step
        test_start = test_end - test_size
        if test_start < 0:
            break

        # Purge at boundary so train labels/features don't overlap test horizon
        train_end = test_start - purge
        train_start = train_end - train_size
        if train_start < 0 or train_end <= train_start:
            break

        print(
        f"Run {k+1}/{runs} | "
        f"Train: {dates[train_start]} → {dates[train_end-1]} | "
        f"Test: {dates[test_start]} → {dates[test_end-1]} | "
        f"Train_n={train_end-train_start} | Test_n={test_end-test_start}"
        )

        X_train = X_all[train_start:train_end]
        y_train = y_all[train_start:train_end]
        X_test = X_all[test_start:test_end]
        y_test = y_all[test_start:test_end]

        # OOT distribution (test only)
        dist = _compute_dist(y_test)

        for model_name, model in models.items():
            m = clone(model)
            m.fit(X_train, y_train)
            preds = m.predict(X_test)

            rows.append({
                "run": k + 1,
                "model": model_name,
                "bal_acc": float(balanced_accuracy_score(y_test, preds)),
                "f1": float(f1_score(y_test, preds, zero_division=0)),
                "acc": float(accuracy_score(y_test, preds)),
                **dist,
                "train_n": int(len(y_train)),
                "train_start": dates[train_start] if dates is not None else train_start,
                "train_end": dates[train_end - 1] if dates is not None else train_end - 1,
                "test_start": dates[test_start] if dates is not None else test_start,
                "test_end": dates[test_end - 1] if dates is not None else test_end - 1,
                "train_years": train_years,
                "horizon_days": horizon_days,
                "n_features": len(safe_feature_cols),
            })

    return pd.DataFrame(rows)

# -----------------------------
# Run grid (feature sets x horizon x train_years, etc.)
# -----------------------------
column_sets = [daily_cols, min_cols, daily_cols + min_cols]
names = ["daily", "minute", "daily+minute"]

returns = [1]#, 3, 5]  # add 3,5,etc later
train_years_grid = [5]#[3, 5, 7]  # could be [3,4,5,6]
runs = 50
test_days = 5
step_days = 5

results_all = []

for feature_cols, feat_name in zip(column_sets, names):
    for r in returns:
        target_col = f"Return_{r}"

        for train_years in train_years_grid:
            df_scores = walkback_runs(
                df=df_main,
                feature_cols=feature_cols,
                target_col=target_col,
                date_col="Date",
                train_years=train_years,
                test_days=test_days,
                step_days=step_days,
                runs=runs,
                horizon_days=r,
                purge_days=r,   # purge = horizon (safe default)
                fill_inf=0.0,
            )

            df_scores["feature_set"] = feat_name
            df_scores["horizon"] = r

            results_all.append(df_scores)

results_df = pd.concat(results_all, ignore_index=True)

# -----------------------------
# Simple summaries you’ll actually use
# -----------------------------
# mean/std across the 20 OOT runs (deployment-aligned)
summary_df = (
    results_df
    .groupby(["feature_set", "horizon", "train_years", "model"], as_index=False)
    .agg(
        bal_acc_mean=("bal_acc", "mean"),
        bal_acc_std=("bal_acc", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        acc_mean=("acc", "mean"),
        acc_std=("acc", "std"),
        test_pos_frac_mean=("test_pos_frac", "mean"),
        test_pos_frac_min=("test_pos_frac", "min"),
        test_pos_frac_max=("test_pos_frac", "max"),
        n_runs=("run", "nunique"),
    )
    .sort_values(["feature_set", "horizon", "bal_acc_mean"], ascending=[True, True, False])
)


Run 1/50 | Train: 2021-01-14 → 2025-12-11 | Test: 2025-12-15 → 2025-12-19 | Train_n=1225 | Test_n=5
Run 2/50 | Train: 2021-01-07 → 2025-12-04 | Test: 2025-12-08 → 2025-12-12 | Train_n=1225 | Test_n=5
Run 3/50 | Train: 2020-12-30 → 2025-11-25 | Test: 2025-12-01 → 2025-12-05 | Train_n=1225 | Test_n=5
Run 4/50 | Train: 2020-12-21 → 2025-11-18 | Test: 2025-11-20 → 2025-11-26 | Train_n=1225 | Test_n=5
Run 5/50 | Train: 2020-12-14 → 2025-11-11 | Test: 2025-11-13 → 2025-11-19 | Train_n=1225 | Test_n=5
Run 6/50 | Train: 2020-12-07 → 2025-11-04 | Test: 2025-11-06 → 2025-11-12 | Train_n=1225 | Test_n=5
Run 7/50 | Train: 2020-11-30 → 2025-10-28 | Test: 2025-10-30 → 2025-11-05 | Train_n=1225 | Test_n=5
Run 8/50 | Train: 2020-11-19 → 2025-10-21 | Test: 2025-10-23 → 2025-10-29 | Train_n=1225 | Test_n=5
Run 9/50 | Train: 2020-11-12 → 2025-10-14 | Test: 2025-10-16 → 2025-10-22 | Train_n=1225 | Test_n=5
Run 10/50 | Train: 2020-11-05 → 2025-10-07 | Test: 2025-10-09 → 2025-10-15 | Train_n=1225 | Test_n=5

KeyboardInterrupt: 

In [13]:
# make sure test_pos_frac is exactly one of the 6 discrete buckets
results_df = results_df.copy()
results_df["test_pos_frac_bucket"] = (
    (results_df["test_pos_frac"] * 5).round().astype(int).clip(0, 5) / 5
)

balacc_by_posfrac = (
    results_df
    .groupby(["feature_set", "horizon", "train_years", "model", "test_pos_frac_bucket"], as_index=False)
    .agg(bal_acc_mean=("bal_acc", "mean"), n=("run", "count"))
)

pivot_balacc = (
    balacc_by_posfrac
    .pivot_table(
        index=["feature_set", "horizon", "train_years", "model"],
        columns="test_pos_frac_bucket",
        values="bal_acc_mean",
        aggfunc="mean",
    )
    .reindex(columns=[0, 0.2, 0.4, 0.6, 0.8, 1.0])  # force desired column order
    .reset_index()
)

pivot_balacc.columns.name = None
pivot_balacc

,feature_set,horizon,train_years,model,0.0,0.2,0.4,0.6,0.8,1.0
0,daily,1,5,random_forest,0.0,0.625000,0.553571,0.612500,0.482143,0.16
1,daily,1,5,xgboost,0.4,0.375000,0.595238,0.604167,0.553571,0.40
2,daily+minute,1,5,random_forest,0.0,0.583333,0.529762,0.570833,0.410714,0.68
3,daily+minute,1,5,xgboost,0.0,0.541667,0.553571,0.570833,0.464286,0.64
4,minute,1,5,random_forest,0.0,0.500000,0.553571,0.562500,0.607143,0.84
5,minute,1,5,xgboost,0.2,0.583333,0.577381,0.520833,0.571429,0.56


In [23]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    balanced_accuracy_score,
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)

# -----------------------------
# Models
# -----------------------------
models = {
    "xgboost": XGBClassifier(n_estimators=400, random_state=42, n_jobs=-1),
    "random_forest": RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1),
}

# -----------------------------
# Helpers
# -----------------------------
def _compute_dist(y):
    """Distribution stats for y in {0,1}."""
    n = int(len(y))
    n_pos = int((y == 1).sum())
    n_neg = int((y == 0).sum())
    return {
        "test_n": n,
        "test_pos_n": n_pos,
        "test_neg_n": n_neg,
        "test_pos_frac": (n_pos / n) if n else np.nan,
        "test_neg_frac": (n_neg / n) if n else np.nan,
    }

def walkback_runs(
    df,
    feature_cols,
    target_col,
    *,
    date_col="Date",
    train_years=6,
    test_days=5,
    step_days=5,
    runs=20,
    horizon_days=1,        # r (used for purge)
    purge_days=None,       # defaults to horizon_days
    fill_inf=0.0,
):
    """
    Deployment-aligned evaluation:
      - For each run, take a 5-day OOT test window stepping back by 5 days.
      - Train on the prior N years (fixed-length window) ending right before test.
      - Purge 'purge_days' from the end of train to avoid overlap leakage for forward-return labels.
      - Score ONLY on the OOT test window (distribution + metrics).
    Returns: long DataFrame with one row per (feature_set/run/model).
    """
    dfw = df.sort_values("Date").reset_index(drop=True).copy()

    # Drop any accidental return cols from features (belt+suspenders)
    safe_feature_cols = [c for c in feature_cols if "Return" not in c]

    # Basic numeric cleaning
    dfw[safe_feature_cols] = dfw[safe_feature_cols].replace([np.inf, -np.inf], fill_inf)

    n = len(dfw)
    train_size = 245 * int(train_years)
    test_size = int(test_days)
    step = int(step_days)
    purge = int(purge_days) if purge_days is not None else int(horizon_days)

    X_all = dfw[safe_feature_cols].to_numpy()
    #y_all = _to_binary(dfw[target_col].to_numpy())
    y_all = dfw[target_col].to_numpy()
    dates = dfw[date_col].to_numpy() if date_col in dfw.columns else None

    rows = []

    for k in range(runs):
        test_end = n - k * step
        test_start = test_end - test_size
        if test_start < 0:
            break

        train_end = test_start - purge
        train_start = train_end - train_size
        if train_start < 0 or train_end <= train_start:
            break

        print(
            f"Run {k+1}/{runs} | "
            f"Train: {dates[train_start]} → {dates[train_end-1]} | "
            f"Test: {dates[test_start]} → {dates[test_end-1]} | "
            f"Train_n={train_end-train_start} | Test_n={test_end-test_start}"
        )

        X_train = X_all[train_start:train_end]
        y_train = y_all[train_start:train_end]
        X_test  = X_all[test_start:test_end]
        y_test  = y_all[test_start:test_end]

        dist = _compute_dist(y_test)
        single_class_test = (np.unique(y_test).size < 2)

        for model_name, model in models.items():
            m = clone(model)
            m.fit(X_train, y_train)

            preds = m.predict(X_test)

            # probabilities if available (for confidence metrics)
            proba = None
            if hasattr(m, "predict_proba"):
                proba = m.predict_proba(X_test)[:, 1]
            elif hasattr(m, "decision_function"):
                s = m.decision_function(X_test)
                # squash to (0,1) so confidence metrics work consistently
                proba = 1.0 / (1.0 + np.exp(-s))

            # confidence/coverage metrics (optional but useful)
            topk_acc = np.nan
            topk_cov = np.nan
            if proba is not None and len(proba) > 0:
                conf = np.abs(proba - 0.5)
                # top 40% by confidence (with 5 samples, this is ~2 samples)
                q = np.quantile(conf, 0.60)
                sel = conf >= q
                topk_cov = float(sel.mean())
                topk_acc = float((preds[sel] == y_test[sel]).mean()) if sel.any() else np.nan

            rows.append({
                "run": k + 1,
                "model": model_name,

                # core metrics
                "bal_acc": float(balanced_accuracy_score(y_test, preds)),
                "acc": float(accuracy_score(y_test, preds)),
                "sign_acc": 2 * float(accuracy_score(y_test, preds)) - 1,
                "mcc": float(matthews_corrcoef(y_test, preds)),

                # only meaningful if test has both classes
                "f1": np.nan if single_class_test else float(f1_score(y_test, preds, zero_division=0)),
                "precision": np.nan if single_class_test else float(precision_score(y_test, preds, zero_division=0)),
                "recall": np.nan if single_class_test else float(recall_score(y_test, preds, zero_division=0)),

                # confidence-conditioned performance (if proba/decision_function exists)
                "top40_acc": topk_acc,
                "top40_cov": topk_cov,

                **dist,

                "train_n": int(len(y_train)),
                "train_start": dates[train_start] if dates is not None else train_start,
                "train_end": dates[train_end - 1] if dates is not None else train_end - 1,
                "test_start": dates[test_start] if dates is not None else test_start,
                "test_end": dates[test_end - 1] if dates is not None else test_end - 1,
                "train_years": train_years,
                "horizon_days": horizon_days,
                "n_features": len(safe_feature_cols),
            })

    return pd.DataFrame(rows)

# -----------------------------
# Run grid (feature sets x horizon x train_years, etc.)
# -----------------------------
column_sets = [daily_cols, min_cols, daily_cols + min_cols]
names = ["daily", "minute", "daily+minute"]

returns = [3]#, 3, 5]  # add 3,5,etc later
train_years_grid = [5]#[3, 5, 7]  # could be [3,4,5,6]
runs = 50
test_days = 5
step_days = 5

results_all = []

for feature_cols, feat_name in zip(column_sets, names):
    for r in returns:
        target_col = f"Return_{r}"

        for train_years in train_years_grid:
            df_scores = walkback_runs(
                df=df_main,
                feature_cols=feature_cols,
                target_col=target_col,
                date_col="Date",
                train_years=train_years,
                test_days=test_days,
                step_days=step_days,
                runs=runs,
                horizon_days=r,
                purge_days=r,   # purge = horizon (safe default)
                fill_inf=0.0,
            )

            df_scores["feature_set"] = feat_name
            df_scores["horizon"] = r

            results_all.append(df_scores)

results_df_3d = pd.concat(results_all, ignore_index=True)

Run 1/50 | Train: 2021-01-12 → 2025-12-09 | Test: 2025-12-15 → 2025-12-19 | Train_n=1225 | Test_n=5
Run 2/50 | Train: 2021-01-05 → 2025-12-02 | Test: 2025-12-08 → 2025-12-12 | Train_n=1225 | Test_n=5
Run 3/50 | Train: 2020-12-28 → 2025-11-21 | Test: 2025-12-01 → 2025-12-05 | Train_n=1225 | Test_n=5
Run 4/50 | Train: 2020-12-17 → 2025-11-14 | Test: 2025-11-20 → 2025-11-26 | Train_n=1225 | Test_n=5
Run 5/50 | Train: 2020-12-10 → 2025-11-07 | Test: 2025-11-13 → 2025-11-19 | Train_n=1225 | Test_n=5
Run 6/50 | Train: 2020-12-03 → 2025-10-31 | Test: 2025-11-06 → 2025-11-12 | Train_n=1225 | Test_n=5
Run 7/50 | Train: 2020-11-24 → 2025-10-24 | Test: 2025-10-30 → 2025-11-05 | Train_n=1225 | Test_n=5
Run 8/50 | Train: 2020-11-17 → 2025-10-17 | Test: 2025-10-23 → 2025-10-29 | Train_n=1225 | Test_n=5
Run 9/50 | Train: 2020-11-10 → 2025-10-10 | Test: 2025-10-16 → 2025-10-22 | Train_n=1225 | Test_n=5
Run 10/50 | Train: 2020-11-03 → 2025-10-03 | Test: 2025-10-09 → 2025-10-15 | Train_n=1225 | Test_n=5

In [34]:
results_df['signed_acc'] = 2 * results_df['acc'] - 1

In [40]:
bins = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
idx  = ["feature_set", "horizon", "train_years", "model"]

# ensure exact bin values (5-day test window)
results_df = results_df.copy()
results_df["test_pos_frac"] = ((results_df["test_pos_frac"] * 5).round() / 5).clip(0, 1)

g = (
    results_df
    .groupby(idx + ["test_pos_frac"], as_index=False)
    .agg(
        signed_acc_mean=("signed_acc", "mean"),
        n_runs=("run", "count"),
    )
)

wide = (
    g.pivot(index=idx, columns="test_pos_frac", values=["signed_acc_mean", "n_runs"])
     .reindex(columns=bins, level=1)
)

wide.columns = [f"{metric}_{frac:g}" for metric, frac in wide.columns]
wide = wide.reset_index()
wide.round(3)


,feature_set,horizon,train_years,model,signed_acc_mean_0,signed_acc_mean_0.2,signed_acc_mean_0.4,signed_acc_mean_0.6,signed_acc_mean_0.8,signed_acc_mean_1,n_runs_0,n_runs_0.2,n_runs_0.4,n_runs_0.6,n_runs_0.8,n_runs_1
0,daily,1,5,random_forest,-1.0,-0.200,0.029,0.20,-0.143,-0.68,1.0,3.0,14.0,20.0,7.0,5.0
1,daily,1,5,xgboost,-0.2,-0.200,0.143,0.18,0.086,-0.20,1.0,3.0,14.0,20.0,7.0,5.0
2,daily+minute,1,5,random_forest,-1.0,-0.333,-0.086,0.20,0.143,0.36,1.0,3.0,14.0,20.0,7.0,5.0
3,daily+minute,1,5,xgboost,-1.0,-0.067,0.000,0.12,-0.029,0.28,1.0,3.0,14.0,20.0,7.0,5.0
4,minute,1,5,random_forest,-1.0,-0.200,0.000,0.22,0.429,0.68,1.0,3.0,14.0,20.0,7.0,5.0
5,minute,1,5,xgboost,-0.6,0.067,0.086,0.08,0.143,0.12,1.0,3.0,14.0,20.0,7.0,5.0


In [52]:
bins = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
idx  = ["feature_set", "horizon", "train_years", "model"]

dfs = [results_df, results_df_3d]
final_df = pd.DataFrame()

for df in dfs:

    df = df.copy()
    # ensure exact bin values (5-day test window)
    df = df.copy()
    df["test_pos_frac"] = ((df["test_pos_frac"] * 5).round() / 5).clip(0, 1)

    g = (
        df
        .groupby(idx + ["test_pos_frac"], as_index=False)
        .agg(
            m=("signed_acc", "mean"),
            n=("run", "count"),
        )
    )

    wide = (
        g.pivot(index=idx, columns="test_pos_frac", values=["m", "n"])
        .reindex(columns=bins, level=1)
    )

    wide.columns = [f"{metric}_{frac:g}" for metric, frac in wide.columns]
    wide = wide.reset_index()
    column_order = ['feature_set', 'horizon', 'train_years', 'model', 'm_0', 'n_0', 'm_0.2', 
                    'n_0.2', 'm_0.4', 'n_0.4', 'm_0.6', 'n_0.6', 'm_0.8', 'n_0.8', 'm_1', 'n_1']
    
    if final_df is None:
        final_df = wide[column_order].round(3).copy()
    else:
        final_df = pd.concat(
            [final_df, wide[column_order].round(3).copy()],
            ignore_index=True
        )



,feature_set,horizon,train_years,model,m_0,n_0,m_0.2,n_0.2,m_0.4,n_0.4,m_0.6,n_0.6,m_0.8,n_0.8,m_1,n_1
0,daily,1,5,random_forest,-1.000,1.0,-0.200,3.0,0.029,14.0,0.200,20.0,-0.143,7.0,-0.680,5.0
1,daily,1,5,xgboost,-0.200,1.0,-0.200,3.0,0.143,14.0,0.180,20.0,0.086,7.0,-0.200,5.0
2,daily+minute,1,5,random_forest,-1.000,1.0,-0.333,3.0,-0.086,14.0,0.200,20.0,0.143,7.0,0.360,5.0
3,daily+minute,1,5,xgboost,-1.000,1.0,-0.067,3.0,0.000,14.0,0.120,20.0,-0.029,7.0,0.280,5.0
4,minute,1,5,random_forest,-1.000,1.0,-0.200,3.0,0.000,14.0,0.220,20.0,0.429,7.0,0.680,5.0
5,minute,1,5,xgboost,-0.600,1.0,0.067,3.0,0.086,14.0,0.080,20.0,0.143,7.0,0.120,5.0
6,daily,3,5,random_forest,-0.600,3.0,-0.029,7.0,0.067,6.0,0.267,12.0,0.200,10.0,-0.467,12.0
7,daily,3,5,xgboost,-0.467,3.0,-0.200,7.0,0.200,6.0,0.300,12.0,0.120,10.0,-0.200,12.0
8,daily+minute,3,5,random_forest,-0.867,3.0,-0.200,7.0,0.133,6.0,0.267,12.0,0.440,10.0,-0.167,12.0
9,daily+minute,3,5,xgboost,-0.600,3.0,-0.314,7.0,0.267,6.0,0.267,12.0,0.320,10.0,-0.133,12.0


In [55]:
bins = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
idx  = ["feature_set", "horizon", "train_years", "model"]

dfs = [results_df, results_df_3d]
final_df = None

for df in dfs:
    df = df.copy()

    # bucket to exact 5-day bins
    df["test_pos_frac"] = ((df["test_pos_frac"] * 5).round() / 5).clip(0, 1)

    # overall (distribution-agnostic)
    overall = (
        df.groupby(idx, as_index=False)
          .agg(m_all=("signed_acc", "mean"), n_all=("run", "count"))
    )

    # by-bin
    g = (
        df.groupby(idx + ["test_pos_frac"], as_index=False)
          .agg(m=("signed_acc", "mean"), n=("run", "count"))
    )

    wide = (
        g.pivot(index=idx, columns="test_pos_frac", values=["m", "n"])
         .reindex(columns=bins, level=1)
    )
    wide.columns = [f"{metric}_{frac:g}" for metric, frac in wide.columns]
    wide = wide.reset_index()

    # merge overall into wide
    wide = wide.merge(overall, on=idx, how="left")

    # optional: column order
    column_order = ['feature_set', 'horizon', 'train_years', 'model', 'm_all', 'n_all', 'm_0', 'n_0', 'm_0.2', 
                    'n_0.2', 'm_0.4', 'n_0.4', 'm_0.6', 'n_0.6', 'm_0.8', 'n_0.8', 'm_1', 'n_1']

    wide = wide[column_order].round(3)

    if final_df is None:
        final_df = wide.copy()
    else:
        final_df = pd.concat([final_df, wide.copy()], ignore_index=True)

final_df.sort_values(by=['horizon', 'm_all'], ascending=False)

,feature_set,horizon,train_years,model,m_all,n_all,m_0,n_0,m_0.2,n_0.2,m_0.4,n_0.4,m_0.6,n_0.6,m_0.8,n_0.8,m_1,n_1
10,minute,3,5,random_forest,0.168,50,-0.867,3.0,-0.429,7.0,-0.067,6.0,0.033,12.0,0.520,10.0,0.733,12.0
11,minute,3,5,xgboost,0.088,50,-0.733,3.0,-0.143,7.0,-0.133,6.0,0.067,12.0,0.360,10.0,0.333,12.0
8,daily+minute,3,5,random_forest,0.048,50,-0.867,3.0,-0.200,7.0,0.133,6.0,0.267,12.0,0.440,10.0,-0.167,12.0
9,daily+minute,3,5,xgboost,0.048,50,-0.600,3.0,-0.314,7.0,0.267,6.0,0.267,12.0,0.320,10.0,-0.133,12.0
7,daily,3,5,xgboost,0.016,50,-0.467,3.0,-0.200,7.0,0.200,6.0,0.300,12.0,0.120,10.0,-0.200,12.0
6,daily,3,5,random_forest,-0.040,50,-0.600,3.0,-0.029,7.0,0.067,6.0,0.267,12.0,0.200,10.0,-0.467,12.0
4,minute,1,5,random_forest,0.184,50,-1.000,1.0,-0.200,3.0,0.000,14.0,0.220,20.0,0.429,7.0,0.680,5.0
1,daily,1,5,xgboost,0.088,50,-0.200,1.0,-0.200,3.0,0.143,14.0,0.180,20.0,0.086,7.0,-0.200,5.0
5,minute,1,5,xgboost,0.080,50,-0.600,1.0,0.067,3.0,0.086,14.0,0.080,20.0,0.143,7.0,0.120,5.0
2,daily+minute,1,5,random_forest,0.072,50,-1.000,1.0,-0.333,3.0,-0.086,14.0,0.200,20.0,0.143,7.0,0.360,5.0
